# What is Einsum?

Einsum is short for Einstein summation convention, a notation that allows for concise representation of tensor operations. It is particularly useful in the context of deep learning and scientific computing, where tensors are frequently manipulated. The notation specifies how to sum over indices of tensors, making it easier to express complex operations without writing out all the intermediate steps. 
Einsum can be used to perform a variety of operations, including matrix multiplication, tensor contractions, and more. It is often implemented in libraries like NumPy and TensorFlow, allowing for efficient computation on large datasets.

For demonstration purposes, we will use the Pytorch library to illustrate the use of einsum. Pytorch provides a convenient `torch.einsum` function that allows for flexible tensor operations using the Einstein summation convention.

For einsum, the axis of the matrix are called ranks. For example, a 2D matrix has 2 ranks, a 3D matrix has 3 ranks, and so on. To define an einsum operation, we specify the input tensors and the desired output tensor using the einsum notation. For example, assume we have two tensors `A` and `B`, and we want to perform a matrix multiplication operation. We can use the following einsum notation:

```python
import torch
A = torch.randn(2, 3) 
B = torch.randn(3, 4) 
result = torch.einsum('ik,jk->ij', A, B)
```
In this example, the notation `'ik,jk->ij'` specifies that we want to multiply the tensors `A` and `B` along their second rank (the common dimension) and sum over that dimension, resulting in a new tensor with shape `(2, 4)`. The first part `'ik,jk'` indicates the input tensors, where `i` and `j` are the first and second ranks of `A` and `B`, respectively, and `k` is the common rank that we are summing over. The `->ij` part indicates the desired output shape, which is a tensor with ranks `i` and `j`.

In [1]:
import torch
A = torch.rand(2, 3)  # A 2D tensor with shape (2, 3)
B = torch.rand(3, 4)  # A 2D tensor with shape (3, 4)
result = torch.einsum('ik,kj->ij', A, B)

print("A:\n", A)
print("B:\n", B)
print("Result:\n", result)

A:
 tensor([[0.1035, 0.2013, 0.5887],
        [0.8358, 0.4024, 0.0017]])
B:
 tensor([[0.8088, 0.3215, 0.3449, 0.5616],
        [0.5793, 0.2817, 0.5885, 0.4322],
        [0.4219, 0.5212, 0.1662, 0.2494]])
Result:
 tensor([[0.4487, 0.3968, 0.2520, 0.2920],
        [0.9098, 0.3830, 0.5254, 0.6438]])


If we go further, we can explain the notation with traditional loops. The einsum notation can be thought of as a shorthand for nested loops that iterate over the indices of the tensors. For example, the above operation can be expressed using traditional loops as follows:

```python
import torch
A = torch.randn(2, 3)
B = torch.randn(3, 4)
result = torch.zeros(2, 4)
for i in range(2):
    for j in range(4):
        total = 0
        for k in range(3):  
            total += A[i, k] * B[k, j]
        result[i, j] = total

```
This code performs the same operation as the einsum notation, but it explicitly iterates over the indices of the tensors and computes the result using nested loops. The einsum notation simplifies this process by allowing us to express the operation in a more concise and readable way, without the need for explicit loops.

In [2]:
import torch
# A = torch.rand(2, 3)  # A 2D tensor with shape (2, 3) -- For now we will use previous values
# B = torch.rand(3, 4)  # A 2D tensor with shape (3, 4) -- For now we will use previous values
manual_result = torch.zeros(2, 4)  # Initialize the result tensor with shape (2, 4)
for i in range(2):  # Iterate over the first rank of A
    for j in range(4):  # Iterate over the second rank of B
        total = 0
        for k in range(3):  # Iterate over the common rank (that will be eliminated)
            total += A[i, k] * B[k, j]
        manual_result[i, j] = total
print("Manual Result:\n", manual_result)
print("Are the results equal?", torch.allclose(result, manual_result))

Manual Result:
 tensor([[0.4487, 0.3968, 0.2520, 0.2920],
        [0.9098, 0.3830, 0.5254, 0.6438]])
Are the results equal? True


If we look closely, anything on the left side of `->` symbol defines the input tensors and their ranks, while the right side defines the output tensor and its ranks. The indices that appear in both input tensors are summed over, while those that appear only in one tensor are retained in the output.

We can name the ranks of the input matrices by any letter, the ranks that appear on the right side will be iterated over 

We can verify this further by the following code, which compares the results of the einsum operation and the manual loop implementation. 

In [3]:
import torch

x = torch.tensor([[1,2,3],[4,5,6],[7,8,9]])
y = torch.tensor([[1,2,3],[4,5,6],[7,8,9],[10,11,12]])

result = torch.einsum('ij,lk->ijk', x, y)
print(result)

man_result = torch.zeros((3, 3, 3), dtype=torch.int32)
for i in range(3):
    for j in range(3):
        for k in range(3):
            total = 0
            for l in range(4):
                total += x[i, j] * y[l, k]
            man_result[i, j, k] = total

print(man_result)
print("Are the results equal? ", torch.equal(result, man_result))

tensor([[[ 22,  26,  30],
         [ 44,  52,  60],
         [ 66,  78,  90]],

        [[ 88, 104, 120],
         [110, 130, 150],
         [132, 156, 180]],

        [[154, 182, 210],
         [176, 208, 240],
         [198, 234, 270]]])
tensor([[[ 22,  26,  30],
         [ 44,  52,  60],
         [ 66,  78,  90]],

        [[ 88, 104, 120],
         [110, 130, 150],
         [132, 156, 180]],

        [[154, 182, 210],
         [176, 208, 240],
         [198, 234, 270]]], dtype=torch.int32)
Are the results equal?  True


# Matrix Multiplication as Einsum

The matrix multiplication operation can be expressed using the einsum notation. Let two matrices A and B with dimensions m x n and n x p, respectively. The matrix multiplication using einsum can be expressed as ij,jk->ik, where i is the row index of A, j is the column index of B, and k is the common dimension. The resulting matrix will have dimensions m x p.

In [4]:
import torch
mat_A = torch.rand(3, 4)
mat_B = torch.rand(4, 5)
print("Matrix A:\n", mat_A)
print("Matrix B:\n", mat_B)
result = torch.einsum('ik,kj->ij', mat_A, mat_B)
print("Result of matrix multiplication using einsum:\n", result)

manual_result = torch.zeros(3, 5)  # Initialize the result tensor with shape (3, 5)
for i in range(3):  # Iterate over the first rank of A
    for j in range(5):  # Iterate over the second rank of B
        total = 0
        for k in range(4):  # Iterate over the common rank
            total += mat_A[i, k] * mat_B[k, j]
        manual_result[i, j] = total
print("Manual Result:\n", manual_result)
print("Are the results equal?", torch.allclose(result, manual_result))

Matrix A:
 tensor([[0.8102, 0.7635, 0.6045, 0.6673],
        [0.9580, 0.6802, 0.4859, 0.9336],
        [0.7906, 0.7514, 0.3789, 0.3628]])
Matrix B:
 tensor([[0.0880, 0.0979, 0.8549, 0.5230, 0.4741],
        [0.0362, 0.3096, 0.2383, 0.8287, 0.0027],
        [0.5270, 0.0974, 0.4993, 0.6419, 0.4537],
        [0.8505, 0.1206, 0.7747, 0.3300, 0.4964]])
Result of matrix multiplication using einsum:
 tensor([[0.9851, 0.4551, 1.6933, 1.6647, 0.9917],
        [1.1591, 0.4643, 1.9470, 1.6847, 1.1399],
        [0.6051, 0.3907, 1.3252, 1.3992, 0.7289]])
Manual Result:
 tensor([[0.9851, 0.4551, 1.6933, 1.6647, 0.9917],
        [1.1591, 0.4643, 1.9470, 1.6847, 1.1399],
        [0.6051, 0.3907, 1.3252, 1.3992, 0.7289]])
Are the results equal? True


What if we need to perform matrix multiplication like $AB^T$? We can use the einsum notation to express this operation as well. The notation for this operation would be ij,kj->ik, where i is the row index of A, j is the column index of B, and k is the common dimension. The resulting matrix will have dimensions m x p, where p is the number of rows in B.

`It's all in the ranks and their order in the einsum notation.`


In [5]:
import torch
mat_A = torch.rand(3, 4)
mat_B = torch.rand(5, 4)
print("Matrix A:\n", mat_A)
print("Matrix B:\n", mat_B)
result = torch.einsum('ik,jk->ij', mat_A, mat_B) # this line is changed
print("Result of matrix multiplication using einsum:\n", result)

manual_result = torch.zeros(3, 5) 
for i in range(3): 
    for j in range(5): 
        total = 0
        for k in range(4): 
            total += mat_A[i, k] * mat_B[j, k] # this line is changed
        manual_result[i, j] = total

print("Manual Result:\n", manual_result)
print("Are the results equal?", torch.allclose(result, manual_result))

Matrix A:
 tensor([[0.4283, 0.5287, 0.9998, 0.8958],
        [0.4250, 0.3119, 0.1733, 0.7042],
        [0.6754, 0.6312, 0.0411, 0.2712]])
Matrix B:
 tensor([[0.0962, 0.6497, 0.2517, 0.8128],
        [0.8964, 0.0284, 0.4396, 0.2947],
        [0.0986, 0.3610, 0.7460, 0.9138],
        [0.2772, 0.8999, 0.1198, 0.8715],
        [0.2152, 0.7806, 0.0533, 0.7397]])
Result of matrix multiplication using einsum:
 tensor([[1.3646, 1.1024, 1.7975, 1.4950, 1.2208],
        [0.8595, 0.6736, 0.9273, 1.0329, 0.8650],
        [0.7059, 0.7213, 0.5730, 0.9965, 0.8409]])
Manual Result:
 tensor([[1.3646, 1.1024, 1.7975, 1.4950, 1.2208],
        [0.8595, 0.6736, 0.9273, 1.0329, 0.8650],
        [0.7059, 0.7213, 0.5730, 0.9965, 0.8409]])
Are the results equal? True


What if we want to perform multiple matrix multiplications in a single einsum operation? We can do that too! For example, if we have three matrices `A`, `B`, and `C` with dimensions `m x n`, `n x p`, and `p x q`, respectively, we can express the operation as `ij,jk,kl->il`. This notation indicates that we want to multiply A and B along their common dimension j, then multiply the result with C along its common dimension k, resulting in a new tensor with dimensions m x q.

In [6]:
import torch

mat_A = torch.rand(3, 4)
mat_B = torch.rand(4, 5)
mat_C = torch.rand(5, 6)
print("Matrix A:\n", mat_A)
print("Matrix B:\n", mat_B)
print("Matrix C:\n", mat_C)
result = torch.einsum('ij,jk,kl->il', mat_A, mat_B, mat_C)
print("Result of matrix multiplication using einsum:\n", result)

manual_result = torch.zeros(3, 6) 
for i in range(3): 
    for l in range(6): 
        total = 0
        for j in range(4): 
            for k in range(5):
                total += mat_A[i, j] * mat_B[j, k] * mat_C[k, l]
        manual_result[i, l] = total
print("Manual Result:\n", manual_result)
print("Are the results equal?", torch.allclose(result, manual_result))

Matrix A:
 tensor([[0.9158, 0.0329, 0.7578, 0.2728],
        [0.5902, 0.0254, 0.1835, 0.9563],
        [0.4584, 0.1760, 0.8325, 0.9165]])
Matrix B:
 tensor([[0.5802, 0.8989, 0.7246, 0.1961, 0.8336],
        [0.7007, 0.0137, 0.9008, 0.3112, 0.3452],
        [0.5551, 0.0061, 0.2768, 0.6896, 0.6989],
        [0.3256, 0.2638, 0.9012, 0.9453, 0.6339]])
Matrix C:
 tensor([[0.8345, 0.0013, 0.5742, 0.6248, 0.9268, 0.2735],
        [0.0326, 0.8389, 0.2841, 0.7283, 0.5247, 0.5789],
        [0.2210, 0.8515, 0.5428, 0.1989, 0.8223, 0.3604],
        [0.6748, 0.7878, 0.4568, 0.1452, 0.4074, 0.4046],
        [0.0792, 0.9845, 0.2977, 0.9059, 0.6628, 0.9505]])
Result of matrix multiplication using einsum:
 tensor([[1.9426, 3.9534, 2.3731, 3.0281, 3.7773, 3.0229],
        [1.8490, 3.9450, 2.3020, 2.6124, 3.5384, 2.7981],
        [2.5198, 4.7032, 2.8901, 3.1928, 4.3950, 3.4226]])
Manual Result:
 tensor([[1.9426, 3.9534, 2.3731, 3.0281, 3.7773, 3.0229],
        [1.8490, 3.9450, 2.3020, 2.6124, 3.5384, 2.7

# Scalar Vector Multiplication as Einsum

Let, A is a vector with dimensions n and B is a scalar. The scalar vector multiplication using einsum can be expressed as i,j->i, where i is the index of the vector A and j is the scalar B. The resulting vector will have dimensions n.


In [7]:
import torch

vector_A = torch.rand(5)
B = torch.rand(1)
print("Vector A:\n", vector_A)
print("Scalar B:\n", B)
result = torch.einsum('i,j->i', vector_A, B)
print("Result of scalar vector multiplication using einsum:\n", result)

manual_result = torch.zeros(5) 
for i in range(5):
    manual_result[i] = vector_A[i] * B[0]
print("Manual Result:\n", manual_result)
print("Are the results equal?", torch.allclose(result, manual_result))

Vector A:
 tensor([0.3725, 0.4582, 0.1562, 0.7536, 0.2678])
Scalar B:
 tensor([0.8031])
Result of scalar vector multiplication using einsum:
 tensor([0.2992, 0.3679, 0.1254, 0.6052, 0.2151])
Manual Result:
 tensor([0.2992, 0.3679, 0.1254, 0.6052, 0.2151])
Are the results equal? True


# Convert Einsum to CUDA Kernel with Tiling

When implementing einsum operations in CUDA, we can optimize the performance by using tiling. Tiling is a technique that allows us to break down the computation into smaller chunks, which can be processed in parallel. This is particularly useful for large tensors, where the memory footprint can be reduced and the performance can be improved.

The ranks that appear on the right side of the `->` symbol can be used to tile to improve the performance. Let's explore the matrix multiplication example. Matrix multiplication einsum can be written as `ij,jk->ik`. 
The kernel can be implemented to perform operation on the i and k ranks, that is, the kernel will be launched to iterate over the i and k ranks, while the j rank will be summed over in each kernel. Since we are iterating over the i and k ranks, we can tile the j rank to improve the performance. The tiling will allow us to perform the operation on smaller chunks of the j rank, which can be processed in parallel. This will reduce the memory footprint and improve the performance of the matrix multiplication operation.

```C++
__global__ void matmul_tiled_kernel(float* A, float* B, float* C, int M, int N, int P) {
    __shared__ float tileA_s[TILE_SIZE][TILE_SIZE];
    __shared__ float tileB_s[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y; //RANK i
    int col = blockIdx.x * TILE_SIZE + threadIdx.x; //RANK k

    float sum = 0.0f;
    for (int t = 0; t < (N + TILE_SIZE - 1) / TILE_SIZE; ++t) { // Loop over tiles 
        if (row < M && t * TILE_SIZE + threadIdx.x < N) // Load A tile
            tileA_s[threadIdx.y][threadIdx.x] = A[row * N + t * TILE_SIZE + threadIdx.x];
        else
            tileA_s[threadIdx.y][threadIdx.x] = 0.0f;

        if (t * TILE_SIZE + threadIdx.y < N && col < P) // Load B tile
            tileB_s[threadIdx.y][threadIdx.x] = B[(t * TILE_SIZE + threadIdx.y) * P + col];
        else
            tileB_s[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();

        for (int j = 0; j < TILE_SIZE; ++j) // Iterate over the RANK j
            sum += tileA_s[threadIdx.y][j] * tileB_s[j][threadIdx.x];

        __syncthreads();
    }

    if (row < M && col < P)
        C[row * P + col] = sum;
}
```

In [8]:
!nvcc -arch sm_86 -o matmul matmul.cu --run
!nsys profile --stats=true --force-overwrite true -o matmul_profile.nsys-rep ./matmul

Max error between CPU and GPU results: 4.57764e-05
Matrix multiplication passed correctness check!
         This may increase runtime overhead and the likelihood of false
         dependencies across CUDA Streams. If you wish to avoid this, please
         disable the feature with --cuda-event-trace=false.
Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Max error between CPU and GPU results: 4.57764e-05
Matrix multiplication passed correctness check!
Generating '/tmp/nsys-report-0b79.qdstrm'
[1/8] [========================100%] matmul_profile.nsys-rep
[2/8] [========================100%] matmul_profile.sqlite
[3/8] Executing 'nvtx_sum' stats report
SKIPPED: /home/spire-zk/PMPP_notebooks/matmul_profile.sqlite does not contain NV Tools Extension (NVTX) data.
[4/8] Executing 'osrt_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)     Med (ns)    Min (ns)   Max (ns)    StdDev (ns)            Name    

Now let's dive deeper into designing a tiling kernel based on the einsum notation. Consider two 2D matrices A and B with dimensions (m, n) and (n, p) respectively. The einsum notation that we want to optimize is `ij, jk -> ik`. This notation indicates that we want to perform a tensor contraction over the second rank (j) of both matrices A and B, resulting in a new matrix C with dimensions (m, p).

A : Matrix A with dimensions (m, n)

B : Matrix B with dimensions (n, p)

C : Resulting matrix with dimensions (m, p)

Einsum notation: `ij, jk -> ik`

Computation: 
$$
C_{i,k} = \sum_j A_{i,j} \cdot B_{j,k}
$$

Step 1: Analyze the computation and identify the parallelizable dimensions. In this case, the output ranks `i` and `k` can be parallelized across multiple threads.

Step 2: Implement the tiling strategy. We will tile the `j` dimension to take advantage of shared memory and reduce global memory accesses.

Step 3: Write the CUDA kernel to perform the tiled matrix multiplication based on the einsum notation.

First calculate the ranks of the thread that will be iterated over in the kernel. The ranks are defined as follows:
```C++
unsigned int i = blockIdx.x * TILE_SIZE + threadIdx.x; // RANK i
unsigned int k = blockIdx.y * TILE_SIZE + threadIdx.y; // RANK k
```
Now we can define the shared memory tiles for A and B. The tiles will be defined as follows:
```C++
__shared__ float tileA_s[TILE_SIZE][TILE_SIZE];
__shared__ float tileB_s[TILE_SIZE][TILE_SIZE];
```

Next, we will load the tiles into shared memory. The loading will be done in a loop that iterates over the `j` rank. The loading will be done as follows:
```C++
for (int t = 0; t < (N + TILE_SIZE - 1) / TILE_SIZE; ++t) {
    // Load A tile
    // Load B tile
    // Synchronize threads
    // Perform the multiplication and summation
}
```


# How Einsum improves matrix multiplication using tiling?

1. The einsum notation allows us to express the matrix multiplication operation in a concise and readable way, making it easier to understand and implement.
2. Tiling allows us to break down the computation into smaller chunks, which can be processed in parallel, reducing the memory footprint and improving performance.
3. The use of shared memory in the tiling kernel allows for faster access to the data, reducing the number of global memory accesses and improving the overall performance of the matrix multiplication operation.
4. The einsum notation allows us to express complex tensor operations in a single line, making it easier to implement and maintain the code.
5. The tiling kernel can be optimized further by adjusting the tile size based on the hardware capabilities, allowing for better utilization of the GPU resources.

# Other Einsum Examples

*Summation* : 
```python
torch.einsum('ij->', A)  
```
*Row Sum* :
```python
torch.einsum('ij->i', A)
```
*Column Sum* :
```python
torch.einsum('ij->j', A)
```

In [9]:
import torch
A = torch.rand(2, 3)
result = torch.einsum('ij->', A)
print("A:\n", A)
print("Result:\n", result)

row_sum = torch.einsum('ij->i', A)
print("Row Sum:\n", row_sum)

col_sum = torch.einsum('ij->j', A)
print("Column Sum:\n", col_sum)

A:
 tensor([[0.1708, 0.9463, 0.3404],
        [0.6187, 0.9310, 0.3170]])
Result:
 tensor(3.3241)
Row Sum:
 tensor([1.4574, 1.8667])
Column Sum:
 tensor([0.7895, 1.8773, 0.6574])


*Element-wise multiplication* :
```python
torch.einsum('ij,ij->ij', A, B)
```

In [10]:
import torch
mat_A = torch.rand(3, 4)
mat_B = torch.rand(3, 4)
print("Matrix A:\n", mat_A)
print("Matrix B:\n", mat_B)
result = torch.einsum('ij,ij->ij', mat_A, mat_B)
print("Element-wise multiplication result:\n", result)

manual_result = torch.zeros(3, 4)  
for i in range(3):  
    for j in range(4): 
        manual_result[i, j] = mat_A[i, j] * mat_B[i, j]
print("Manual Result:\n", manual_result)
print("Are the results equal?", torch.allclose(result, manual_result))

Matrix A:
 tensor([[0.3374, 0.5979, 0.5349, 0.0684],
        [0.1192, 0.6102, 0.6307, 0.8742],
        [0.2921, 0.6178, 0.3611, 0.5462]])
Matrix B:
 tensor([[0.4988, 0.2276, 0.1606, 0.8097],
        [0.8410, 0.1971, 0.3400, 0.9579],
        [0.5583, 0.3625, 0.2967, 0.8757]])
Element-wise multiplication result:
 tensor([[0.1683, 0.1361, 0.0859, 0.0554],
        [0.1002, 0.1203, 0.2144, 0.8373],
        [0.1631, 0.2239, 0.1071, 0.4783]])
Manual Result:
 tensor([[0.1683, 0.1361, 0.0859, 0.0554],
        [0.1002, 0.1203, 0.2144, 0.8373],
        [0.1631, 0.2239, 0.1071, 0.4783]])
Are the results equal? True


# Cooley-Tukey FFT steps

In this step we will use einsum to perform cooley-tukey FFT steps. The Cooley-Tukey algorithm is a divide-and-conquer algorithm for computing the discrete Fourier transform (DFT) and its inverse. It is the most common algorithm for computing the DFT, and it is widely used in many applications, including signal processing, image processing, and data compression.

For FFT, a polynomial is represented as a vector of coefficients, where each coefficient corresponds to a power of the variable. The Cooley-Tukey algorithm recursively divides the polynomial into smaller sub-polynomials, computes the DFT of each sub-polynomial, and combines the results to obtain the DFT of the original polynomial.

The Cooley-Tukey FFT step can be expressed using the einsum notation. The notation for the Cooley-Tukey FFT step can be written as follows:

$$
E_{0,k_0} = P_{0,k_0,n_1,0} \times X_{n_1,0}
$$

$$
O_{0,k_0} = P_{0,k_0,n_1,0} \times X_{n_1,1}
$$

$$
T_{k_0} = P_{0,k_0,0,1} \times O_{0,k_0}
$$

$$
Y_{0,k_0} = E_{0,k_0} + T_{k_0}
$$

$$
Y_{1,k_0} = E_{0,k_0} - T_{k_0}
$$


Let the input vector be reshaped and decomposed as part of the Cooley-Tukey FFT algorithm using tensor contractions and einsum notation. Each term in the following equations has the following meaning:

- $X$: Input signal, reshaped as a 2D array according to the Cooley-Tukey decomposition.
- $P$: DFT matrix (roots of unity powers), typically precomputed and indexed based on the decomposition.
- $E_{0,k_0}$: Even-indexed partial FFT result for the first half of the decomposition.
- $O_{0,k_0}$: Odd-indexed partial FFT result for the second half of the decomposition.
- $T_{k_0}$: Twiddle factor multiplied with the odd result $O_{0,k_0}$.
- $Y_{0,k_0}$: Final FFT result for the even index after combining.
- $Y_{1,k_0}$: Final FFT result for the odd index after combining.


The steps follow the divide-and-conquer approach of the Cooley-Tukey FFT:

1. Divide input $X into even and odd components.
2. Apply smaller DFTs to even and odd parts:
   - $ E_{0,k_0} = P_{0,k_0,n_1,0} \cdot X_{n_1,0} $
   - $ O_{0,k_0} = P_{0,k_0,n_1,0} \cdot X_{n_1,1} $
3. Apply twiddle factors:
   - $ T_{k_0} = P_{0,k_0,0,1} \cdot O_{0,k_0} $
4. Combine results to get the final FFT outputs:
   - $ Y_{0,k_0} = E_{0,k_0} + T_{k_0} $
   - $ Y_{1,k_0} = E_{0,k_0} - T_{k_0} $



In [11]:
import numpy as np
import torch

def cooley_tukey_fft_einsum(x):
    """
    Cooley-Tukey FFT implementation using einsum for radix-2 FFT (length must be a power of 2).
    x: torch.Tensor of shape (N,) where N is a power of 2.
    """
    N = x.shape[0]
    assert (N & (N - 1)) == 0, "Input length must be power of 2"

    x = x.to(torch.cfloat)  # Ensure complex dtype

    n1 = N // 2

    # Reshape input into (n1, 2)
    x_reshaped = x.view(n1, 2)

    # Create small DFT matrix for n1 size FFT
    k0 = torch.arange(n1).reshape(-1, 1)
    n = torch.arange(n1).reshape(1, -1)
    omega_n1 = torch.exp(-2j * np.pi * k0 * n / n1)

    # Even: einsum contraction for 1st column (even indices)
    E = torch.einsum('kn,n->k', omega_n1, x_reshaped[:, 0])

    # Odd: einsum contraction for 2nd column (odd indices)
    O = torch.einsum('kn,n->k', omega_n1, x_reshaped[:, 1])

    # Twiddle factors
    twiddles = torch.exp(-2j * np.pi * torch.arange(n1) / N)

    # Apply twiddle factors to odd results
    T = twiddles * O

    # Final FFT output using Cooley-Tukey combination
    Y_top = E + T
    Y_bottom = E - T

    return torch.cat([Y_top, Y_bottom])

# Example
x = torch.randn(8) + 1j * torch.randn(8)
X = cooley_tukey_fft_einsum(x)

# Compare with PyTorch's FFT
X_ref = torch.fft.fft(x)

print("Our FFT (einsum):", X)
print("Torch FFT       :", X_ref)
print("Max Error       :", torch.max(torch.abs(X - X_ref)))

Our FFT (einsum): tensor([ 3.0903+3.2555j,  0.6125+8.3871j, -7.3624-0.8862j, -0.5542-2.4603j,
        -1.1504-0.5794j,  1.7647-5.2426j, -2.3919-2.8978j,  0.8989-4.1234j])
Torch FFT       : tensor([ 3.0903+3.2555j,  0.6125+8.3871j, -7.3624-0.8862j, -0.5542-2.4603j,
        -1.1504-0.5794j,  1.7647-5.2426j, -2.3919-2.8978j,  0.8989-4.1234j])
Max Error       : tensor(9.5367e-07)
